# Importing Libraries and packages

In [1]:
# Importing Libraries
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.animation as animation
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import scipy as sp
import numpy as np
import pandas as pd
import time
import datetime
import os
import sys
import pyvisa
import threading
import ipywidgets
import nidaqmx
import warnings
from scipy import signal
from collections import deque
from IPython.display import clear_output, display

print(f"All Libraries imported successfully at {datetime.datetime.now()}")

All Libraries imported successfully at 2026-05-06 20:23:17.832027


# Scan Connected Instruments

In [2]:
# Scan Connected Instruments
def scan_instrument():
    rm = pyvisa.ResourceManager() # Open the Resource Manager
    instruments = []
    for address in rm.list_resources(): # Running over all the available addresses in the Resource Manager
        try:
            instrument = rm.open_resource(address)
            responce = instrument.query('*IDN?').strip()
            instruments.append(f"{address:<50} => {responce}")
        except Exception:
            responce = "No responce"
            instruments.append(f"{address:<50} => {responce}")
    # return the list of instruments
    return instruments

for items in scan_instrument():
    print(items)

# Import zhinst.toolkit and connecting to zhinst device
from zhinst.toolkit import Session # Importing Session class from zhinst.toolkit
session = Session('localhost') # Creating a session to connect to the device
print(f"\nVisible Zurich devices: {session.devices.visible()}") # List visible devices
print(f"Connected Zurich devices: {session.devices.connected()}") # List connected devices

print(f"\nInstruments scanned successfully at {datetime.datetime.now()}")

GPIB1::5::INSTR                                    => No responce
GPIB1::9::INSTR                                    => No responce
TCPIP0::192.168.150.190::inst0::INSTR              => No responce
TCPIP0::192.168.150.20::inst0::INSTR               => No responce
TCPIP0::192.168.150.21::inst0::INSTR               => No responce
TCPIP0::localhost::hislip0::INSTR                  => No responce
TCPIP1::192.168.1.15::inst0::INSTR                 => No responce
USB0::0x03EB::0xAFFF::3C6-0B4F40003-0985::0::INSTR => No responce
TCPIP0::192.168.150.124::gpib0,5::INSTR            => No responce
TCPIP0::192.168.150.124::gpib0,25::INSTR           => HEWLETT-PACKARD,34401A,0,11-5-2
TCPIP0::192.168.150.124::gpib0,15::INSTR           => HEWLETT-PACKARD,34401A,0,8-5-2
TCPIP0::192.168.150.124::gpib0,17::INSTR           => No responce
TCPIP0::192.168.150.124::gpib0,3::INSTR            => No responce
TCPIP0::K-M9537A-00010::inst0::INSTR               => No responce
TCPIP0::192.168.150.124::gpib0,8::INS

# For connection at 14T Cryostat

In [2]:
# Open Pyvisa Resource Manager and connect to insturements at 14T cryostat
rm = pyvisa.ResourceManager()
print("Connected instruments list:\naddress handle  : Instrument Connected\n---------------------------------------------")

# lakeshore = rm.open_resource("TCPIP0::192.168.150.121::7777::SOCKET", timeout=5000, write_termination='\n', read_termination='\r\n')
# print(f"lakeshore       : {lakeshore.query('*IDN?'):>20}")

bilt = rm.open_resource("TCPIP0::192.168.150.123::5025::SOCKET", timeout=5000, write_termination='\n', read_termination='\n')
print(f"bilt            : {bilt.query('*IDN?')}")

# ami430 = rm.open_resource("TCPIP0::192.168.150.122::7180::SOCKET", timeout=5000, write_termination='\n', read_termination='\r\n')
# print(f"ami430          : {ami430.query('*IDN?')}")

# dmm15 = rm.open_resource("TCPIP0::192.168.150.124::gpib0,15::INSTR", timeout=5000, write_termination='\n', read_termination='\n')
# print(f"dmm15           : {dmm15.query('*IDN?')}")

dmm25 = rm.open_resource("TCPIP0::192.168.150.124::gpib0,25::INSTR", timeout=5000, write_termination='\n', read_termination='\n')
print(f"dmm25           : {dmm25.query('*IDN?')}")

# sr860 = rm.open_resource("TCPIP0::192.168.150.20::inst0::INSTR", timeout=5000, write_termination='\n', read_termination='\n')
# print(f"sr860           : {sr860.query('*IDN?')}")

# awg1 = rm.open_resource("TCPIP0::A-33622A-01823.local::inst0::INSTR", timeout=5000, write_termination='\n', read_termination='\n')
# print(f"awg1            : {awg1.query('*IDN?')}")

# Import zhinst.toolkit and connecting to zhinst device
from zhinst.toolkit import Session # Importing Session class from zhinst.toolkit
session = Session('localhost') # Creating a session to connect to the device

# mfli4703 = session.connect_device('dev4703')
# print(f"mfli4703        : {mfli4703}")

# mfli4763 = session.connect_device('dev4763')
# print(f"mfli4763        : {mfli4763}")

# mfli7714 = session.connect_device('dev7714')
# print(f"mfli7714        : {mfli7714}")

uhf = session.connect_device('dev2288')
print(f"uhf             : {uhf}")

print(f"\nInstruments connected successfully at {datetime.datetime.now()}")

Connected instruments list:
address handle  : Instrument Connected
---------------------------------------------
bilt            : 2142,"ITEST BE2142C/12V 15mA DC-SOURCE/SN06-013 LC2002 VL446\240"
dmm25           : HEWLETT-PACKARD,34401A,0,11-5-2
uhf             : UHFLI(UHFLI(DIG,MF,PID),dev2288)

Instruments connected successfully at 2026-05-06 20:23:34.261116


# General Functions

In [3]:
# Function for defining the stop button click event handler
def on_stop_clicked(b):
    stop_event.set()
    status_btn.description = "Stopped"
    status_btn.button_style = "danger"
    stop_btn.disabled = True
    stop_btn.close()

# Function for creating path for saving file in a specific folder
def save_file(folder,filename): # It will add date prefix to filename
    """
    It will create the folder if it does not exist 
    and return the full path for saving file in that folder.
    """
    current_path = os.getcwd()
    folder_path = os.path.join(current_path, folder)
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
    filename = f"{datetime.datetime.now().strftime('%y%m%d')}_{filename}"
    file_path = os.path.join(folder_path, filename)
    return file_path

# Function for creating path for loading file from a specific folder
def load_file(folder,filename): # It will not add date prefix to filename
    """
    Please provide the proper filename with extension.
    It will return the full path for loading file from that folder.
    """
    current_path = os.getcwd()
    file_path = os.path.join(current_path, folder, filename)
    return file_path

# Function for saving data in the file
def create_file(data,folder,filename):
    # Create a folder
    directory = os.path.join(os.getcwd(), folder)
    if not os.path.exists(directory):
        os.makedirs(directory)
    full_path = os.path.join(directory, filename) # Path for file saving
    # Create the file
    if not os.path.exists(full_path):
        data.to_csv(full_path, mode='a', header=True, index=False)
    else:
        data.to_csv(full_path, mode='a', header=False, index=False)

# Function for creating array for sweeping
def sweep(vmin, vmax, vstep, vmode): # Modes: up, down, updown, downup, round-up, round-down
    if vmode =='up': # lower to higher value
        sweep = np.arange(vmin,vmax+vstep,vstep)
    elif vmode =='down': # higher to lower value
        sweep = np.arange(vmax,vmin-vstep,-vstep)
    elif vmode =='updown': # lower to higher to lower value
        sweep = np.arange(vmin,vmax+vstep,vstep)
        sweep = np.append(sweep,np.arange(vmax-vstep,vmin-vstep,-vstep))
    elif vmode =='downup': # higher to lower to higher value
        sweep = np.arange(vmax,vmin-vstep,-vstep)
        sweep = np.append(sweep,np.arange(vmin+vstep,vmax+vstep,vstep))
    elif vmode =='round-up': # zero to higher to lower to zero value
        sweep = np.arange(0,vmax+vstep,vstep)
        sweep = np.append(sweep,np.arange(vmax-vstep,vmin-vstep,-vstep))
        sweep = np.append(sweep,np.arange(vmin+vstep,vstep,vstep))
    elif vmode =='round-down': # zero to lower to higher to zero value
        sweep = np.arange(0,vmin-vstep,-vstep)
        sweep = np.append(sweep,np.arange(vmin+vstep,vmax+vstep,vstep))
        sweep = np.append(sweep,np.arange(vmax-vstep,-vstep,-vstep))
    else:
        print('Error: Invalid Sweep Mode')
        sys.exit()
    return sweep

# Function for converting current values to voltage values
def ItoV(i_min,i_max,i_step,R_series):
    '''returns v_min,v_max,v_step'''
    if R_series == 0:
        R_series = 1
    v_min=i_min*R_series
    v_max=i_max*R_series
    v_step=i_step*R_series
    return v_min,v_max,v_step

# Function for converting voltage values to current values
def VtoI(v_min,v_max,v_step,R_series):
    '''returns i_min,i_max,i_step'''
    if R_series == 0:
        R_series = 1
    i_min=v_min/R_series
    i_max=v_max/R_series
    i_step=v_step/R_series
    return i_min,i_max,i_step

# Function for converting XYRT to dVdI
def calculate_dVdI(X,Y,R,I,Gain):
    dVdI_x=X/(I*Gain)
    dVdI_y=Y/(I*Gain)
    dVdI_r=R/(I*Gain)
    return dVdI_x,dVdI_y,dVdI_r

print(f"All Helper functions are defined successfully at {datetime.datetime.now()}")

All Helper functions are defined successfully at 2026-05-06 20:23:36.411072


# Functions for Lakeshore 370/372 (LAN/GPIB)

In [ ]:
# Function to read temperature from Lakeshore 370/372 specific channel
def readtemp_lakeshore(address, channel):
    try:
        temp = lakeshore.query_ascii_values(f"RDGK? {channel}")[0]
    except Exception:
        temp = np.nan
    return temp

# Function to read Temperature Control Mode from Lakeshore 370/372
def readmode_lakeshore(address, model):
    try:
        if model == "370":
            n = lakeshore.query_ascii_values("CMODE?")[0]
            mode_list = ["Closed Loop PID", "Zone Tuning", "Open Loop", "Off"]
            mode = mode_list[int(n)-1]
        elif model == "372":
            n = lakeshore.query("OUTMODE? 0")[0]
            mode_list = ["Off", "Monitor Out", "Open Loop", "Zone", "Still", "Closed Loop PID", "Warm Up"]
            mode = mode_list[int(n)]
    except Exception:
        mode = "Error"
    return mode

# Function to read Heater Range from Lakeshore 370/372
def readrange_lakeshore(address, model):
    range_list = ["off","31.6 uA","100 uA","316 uA","1.00 mA","3.16 mA","10.0 mA","31.6 mA","100 mA"]
    try:
        if model == "370":
            n = lakeshore.query_ascii_values("HTRRNG?")[0]
            range = range_list[int(n)]
        elif model == "372":
            n = lakeshore.query_ascii_values("RANGE? 0")[0]
            range = range_list[int(n)]
    except Exception:
        range = "Error"
    return range

# Function to read Heater PID Parameters from Lakeshore 370/372
def readpid_lakeshore(address, model):
    try:
        if model == "370":
            params = lakeshore.query_ascii_values("PID?")
        elif model == "372":
            params = lakeshore.query_ascii_values("PID? 0")
    except Exception:
        params = [np.nan, np.nan, np.nan]
    return params

# Function to read Heater Status from Lakeshore 370/372
def readheater_lakeshore(address, model):
    try:
        mode = readmode_lakeshore(address, model)
        range = readrange_lakeshore(address, model)
        P, I, D = readpid_lakeshore(address, model)
        status = {"Mode": mode, "Range": range, "P": P, "I": I, "D": D}
    except Exception:
        status = np.nan
    return status

# Function to set Temperature Control Mode on Lakeshore 370/372
def setmode_lakeshore(address, model, mode="Closed Loop PID"):
    try:
        if model == "370":
            mode_list = ["Closed Loop PID", "Zone Tuning", "Open Loop", "Off"]
            if mode not in mode_list:
                print("Invalid Mode. Valid options are:", mode_list)
                return
            n = mode_list.index(mode) + 1
            lakeshore.write(f"CMODE {n}")
        elif model == "372":
            mode_list = ["Off", "Monitor Out", "Open Loop", "Zone", "Still", "Closed Loop PID", "Warm Up"]
            if mode not in mode_list:
                print("Invalid Mode. Valid options are:", mode_list)
                return
            n = mode_list.index(mode)
            lakeshore.write(f"OUTMODE 0,{n},6,1,0,1,60") # <output>,<mode>,<channel>,<powerup enable>,<polarity>,<filter>,<delay>
    except Exception:
        pass

# Function to set Heater Range on Lakeshore 370/372
def setrange_lakeshore(address, model, range="off"): # "off","31.6 uA","100 uA","316 uA","1.00 mA","3.16 mA","10.0 mA","31.6 mA","100 mA"
    try:
        range_list = ["off","31.6 uA","100 uA","316 uA","1.00 mA","3.16 mA","10.0 mA","31.6 mA","100 mA"]
        if range not in range_list:
            print("Invalid Range. Valid options are:", range_list)
            return
        n = range_list.index(range)
        if model == "370":
            lakeshore.write(f"HTRRNG {n}")
        elif model == "372":
            lakeshore.write(f"RANGE 0,{n}") # <channel>,<range>
    except Exception:
        pass

# Function to set Heater PID Parameters on Lakeshore 370/372
def setpid_lakeshore(address, model, P, I, D):
    try:
        if model == "370":
            lakeshore.write(f"PID {P},{I},{D}") # <P>,<I>,<D>
        elif model == "372":
            lakeshore.write(f"PID 0,{P},{I},{D}") # <channel>,<P>,<I>,<D>
    except Exception:
        pass

# Function to set Temperature Setpoint on Lakeshore 370/372
def settemp_lakeshore(address, model, temp):
    try:
        if model == "370":
            lakeshore.write(f"SETP {temp}")
        elif model == "372":
            lakeshore.write(f"SETP 0,{temp}") # <channel>,<setpoint>
    except Exception:
        pass

def stabilizetemp_lakeshore(address, model, temp, tolerance=0.01, timeout=600):
    try:
        settemp_lakeshore(address, model, temp)
        start_time = time.time()
        while time.time() - start_time < timeout:
            current_temp = readtemp_lakeshore(address, 6)
            print(f"Stabilizing {temp} K. Now {current_temp} K.", end="\r")
            if abs(current_temp - temp) <= tolerance*temp:
                print(f"Temperature stabilized at {current_temp} K.")
                return True
            time.sleep(1)
        print(f"Failed to stabilize temperature within {timeout} seconds.")
    except Exception:
        pass

print(f"Lakeshore functions are defined successfully at {datetime.datetime.now()}")

# Functions for AMI 430 Magnet ( not tested)

In [ ]:
# Function to get AMI430 status
def get_ami_status(ami430): # Returns AMI430 state as string
    dummy = ami430.query_ascii_values('STATE?')
    dummy = ami430.query_ascii_values('STATE?')
    ami_state_number = ami430.query_ascii_values('STATE?')
    ami_state_list = ["RAMPING to target", 
                      "HOLDING at target", 
                      "PAUSED", 
                      "Ramping in MANUAL UP mode", 
                      "Ramping in MANUAL DOWN mode", 
                      "ZEROING CURRENT", 
                      "QUENCH DETECTED", 
                      "AT ZERO current", 
                      "Heating persistent switch", 
                      "Cooling persistent switch"]
    ami_state = ami_state_list[int(ami_state_number[0])-1]
    return ami_state

# Function to set AMI430 field and ramp rate units
def set_ami_unit(field_unit, rate_unit, ami430): # Tesla/kiloGauss and per_minute/per_second
    '''
    Tesla or kiloGauss for field_unit. per_minute or per_second for rate_unit.
    1 Tesla = 10 kiloGauss
    '''
    try:
        if field_unit == 'Tesla':
            field_unit_code = 1
        elif field_unit == 'kiloGauss':
            field_unit_code = 0
        if rate_unit == 'per_minute':
            rate_unit_code = 1
        elif rate_unit == 'per_second':
            rate_unit_code = 0
        ami430.write(f"CONFigure:FIELD:UNITS {field_unit_code}")
        ami430.write(f"CONFigure:RAMP:RATE:UNITS {rate_unit_code}")
        print(f'Units set to {field_unit} {rate_unit}')
    except:
        print('Error: Unable to set units. Check AMI430 and retry.')
        sys.exit()

# Function to set AMI430 ramp rate
def set_ami_ramp_rate(rate, ami430): # Set ramp rate in units defined earlier
    """
    For Fincryo Magnet
    Maximum ramp rate for 0-5 Tesla is 1mT/s (60 mT/min)
    Maximum ramp rate for 5-7 Tesla is 0.5mT/s (30 mT/min)
    Maximum ramp rate for 7-9 Tesla is 0.25mT/s (15 mT/min)
    """
    try:
        ami430.write('CONFigure:RAMP:RATE:SEGments 1')
        ami430.write(f'CONFigure:RAMP:RATE:FIELD 1,{rate},9')
    except:
        pass

# Function to set AMI430 magnetic field
def set_ami_field(B_set, ami430): # Set target magnetic field and start ramping
    '''arguments: B_set - target magnetic field, ami430 - AMI430 instrument object'''
    try:
        dummy = ami430.query_ascii_values('STATE?')[0]
        dummy = ami430.query_ascii_values('STATE?')[0]
        ami_state_number = ami430.query_ascii_values('STATE?')[0]
        if ami_state_number in [1, 2, 3]:
            ami430.write(f'CONFigure:FIELD:TARget {B_set}')
            ami430.write(f'RAMP')
    except:
        pass

# Function for logging the magnetic field from AMI430
def log_ami430(ami430): # Temeperature is dummy value here atleast for now
    try:
        """
        Step 1: Create a log file if does not exist
        """
        logfile = os.path.join(os.getcwd(),'Magnet Log',f'{datetime.datetime.now().strftime("%y%m%d")}_AMI430_MagnetLog.log')
        if not os.path.exists(logfile):
            columns = ['Datetime','State','Temperature_K','Field_T','Vmagnet_V','Vsupply_V']
            pd.DataFrame(columns=columns).to_csv(logfile, index=False)
        """
        Step 2: Read parameters from AMI430
        """
        t = datetime.datetime.now().strftime('%y-%m-%d %H:%M:%S')
        dummy = ami430.query_ascii_values('STATE?')[0]
        dummy = ami430.query_ascii_values('STATE?')[0]
        ami_state_number = ami430.query_ascii_values('STATE?')[0]
        Temp = np.nan  # Placeholder for temperature value
        dummy = ami430.query_ascii_values('FIELD:MAGnet?')[0]
        dummy = ami430.query_ascii_values('FIELD:MAGnet?')[0]
        Field = ami430.query_ascii_values('FIELD:MAGnet?')[0]
        dummy = ami430.query_ascii_values('VOLTage:MAGnet?')[0]
        dummy = ami430.query_ascii_values('VOLTage:MAGnet?')[0]
        V_magnet = ami430.query_ascii_values('VOLTage:MAGnet?')[0]
        dummy = ami430.query_ascii_values('VOLTage:SUPPly?')[0]
        dummy = ami430.query_ascii_values('VOLTage:SUPPly?')[0]
        V_supply = ami430.query_ascii_values('VOLTage:SUPPly?')[0]
        row = {"Datetime": t, "State": ami_state_number, "Temperature_K": Temp, "Field_T": Field, "Vmagnet_V": V_magnet, "Vsupply_V": V_supply}
        pd.DataFrame([row]).to_csv(logfile, mode='a', header=False, index=False)
        time.sleep(1)
    except:
        pass

print(f"AMI430 Helper functions are defined successfully at {datetime.datetime.now()}")

# Functions for Bilt BN103 BE2141 BE2142 (LAN)

In [4]:
# Function to read Voltage and Current from Bilt Instruments
def measure_bilt(address,position,channel):
    Voltage = bilt.query(f"I{position};C{channel};MEAS:VOLT?")
    Current = bilt.query(f"I{position};C{channel};MEAS:CURR?")
    return float(Voltage), float(Current)

# Function to set Voltage on Bilt Instruments with Step Ramping
def setvolt_bilt(address,position,channel,voltage,step,steptime=100):
    # address.write(f"I{position};C{channel};TRIG:IN 1") # 0: Exponetial, 1: Ramp, 2: Staircase, 3: Triggered Step, 4: Auto Step
    address.write(f"I{position};C{channel};VOLT:STEP:WIDTH {steptime}") # Step width in milliseconds
    address.write(f"I{position};C{channel};VOLT:STEP:AMPL {step}") # Step size in volts
    address.write(f"I{position};C{channel};VOLT {voltage}")  # Target voltage in volts
    address.write(f"I{position};C{channel};TRIG:IN:INIT")  # Triggering

print(f"BILT functions are defined successfully at {datetime.datetime.now()}")

BILT functions are defined successfully at 2026-05-06 20:23:42.948433


# Functions for Zhurich UHF Lock-in Amplifier and Setting up for the measurement

In [20]:
# Configure the device
SCOPE_LENGTH = 2**16 # Number of samples to acquire
# # 2^1 = 2, 2^2 = 4, 2^3 = 8, 2^4 = 16, 2^5 = 32, 2^6 = 64, 2^7 = 128, 2^8 = 256, 2^9 = 512, 2^10 = 1024, 2^11 = 2048, 2^12 = 4096, 2^13 = 8192
# # 2^14 = 16384, 2^15 = 32768, 2^16 = 65536, 2^17 = 131072, 2^18 = 262144, 2^19 = 524288, 2^20 = 1048576, 2^21 = 2097152, 2^22 = 4194304
# SCOPE_CHANNEL = [0, 1] # Index of the scope channel to use (0 for Channel 1, 1 for Channel 2, [0, 1] for both channels)
SCOPE_TIME = 11 # Specify the sampling rate
# # SCOPE_TIME 0 -> 1.8 GHz, 1 -> 900 MHz, 2 -> 450 MHz, 3 -> 225 MHz, 4 -> 113 MHz, 5 -> 56.2 MHz, 6 -> 28.1 MHz, 7 -> 14 MHz, 8 -> 7.03 MHz
# # SCOPE_TIME 9 -> 3.5 MHz, 10 -> 1.75 MHz, 11 -> 880 kHz, 12 -> 440 kHz, 13 -> 220 kHz, 14 -> 110 kHz, 15 -> 54.9 kHz, 16 -> 27.5 kHz
MIN_NUMBER_OF_RECORDS = 20 # Minimum number of records to acquire
# TRIGGER_HOLDOFF = 0.050 # Trigger holdoff in seconds
# SCOPE_TRIGGER = 0 # Trigger source (0 for Channel 1)

with uhf.set_transaction(): # Sending multiple commands in a single transaction

    # Configure the scope
    uhf.scopes[0].length(SCOPE_LENGTH) # Set the number of samples to acquire.
    uhf.scopes[0].time(SCOPE_TIME) # Sets the scope time base, i.e. sampling rate.
    uhf.scopes[0].single(False) # Puts the Scope into single shot mode.

    # uhf.scopes[0].trigenable(0) # Enable trigger for the scope.
    # uhf.scopes[0].trigchannel(2) # 0: Signal Input 1, 1: Signal Input 2, 2: Trigger Input 1, 3: Trigger Input 2.
    # uhf.scopes[0].trigholdoff(TRIGGER_HOLDOFF) # Sets the trigger holdoff time in seconds.
    # uhf.scopes[0].segments.enable(False)

    uhf.scopes[0].channel(1) # 1 -> Only ch1, 2 -> Only ch2, 3 -> Both channels.
    uhf.scopes[0].channels[0].bwlimit(1) # Enable bandwidth limit for the channel.
    uhf.scopes[0].channels[0].inputselect(0) # Selects the input signal to route to the scope channel.
    uhf.scopes[0].channels[1].bwlimit(1) # Enable bandwidth limit for the channel.
    uhf.scopes[0].channels[1].inputselect(1) # Selects the input signal to route to the scope channel.

    # Configure signal input channel 1
    uhf.sigins[0].on(1) # Enable signal input channel 1
    uhf.sigins[0].ac(0) # 0 -> DC coupling, 1 -> AC coupling.
    uhf.sigins[0].diff(0) # 0 -> Off, 1 -> Inverted, 2 -> Input1 - Input2, 3 -> Input2 - Input1.
    uhf.sigins[0].imp50(0) # 0 -> 1 MOhm input impedance, 1 -> 50 Ohm input impedance.
    uhf.sigins[0].range(1.5) # in Volts (max. 1.5 V)
    uhf.sigins[0].scaling(1)

    # Configure signal input channel 2
    # uhf.sigins[1].on(1) # Enable signal input channel 2
    # uhf.sigins[1].ac(0) # 0 -> DC coupling, 1 -> AC coupling.
    # uhf.sigins[1].diff(0) # 0 -> Off, 1 -> Inverted, 2 -> Input1 - Input2, 3 -> Input2 - Input1.
    # uhf.sigins[1].imp50(0) # 0 -> 1 MOhm input impedance, 1 -> 50 Ohm input impedance.
    # uhf.sigins[1].range(0.1) # in Volts (max. 1.5 V)
    # uhf.sigins[1].scaling(1)
    
scope_module = session.modules.scope # Accessing the scope module.

scope_module.mode(3) # Acquisition mode. 0 for Scope raw data, 1 for averaging mode, 3 for FFT mode.
scope_module.fft.window(1) # FFT window type: 0 > rectangular, 1 > Hann, 2 > Hamming, 3 > Blackman-Harris, 4 > Flat-Top.
scope_module.fft.powercompensation(1) # Enable power compensation for noise floor correction. 0 > no, 1 > yes.
scope_module.fft.power(1) # Enable power units (V^2) for correct noise floor units. 0 > no, 1 > yes.
scope_module.fft.spectraldensity(1) # Enable spectral density (divide by bin width) for correct noise floor units. 0 > no, 1 > yes.

scope_module.historylength(100) # Maximum number of records stored in history buffer (circular buffer).

# scope_module.averager.enable(1) # Enables (1) or disables (0) the averager.
# scope_module.averager.method(1) # Sets the averaging method. 0 for exponential moving average, 1 for uniform averaging.
# scope_module.averager.resamplingmode(0) # Sets tne resampling mode for low sample rate signals. 0 for linear interpolation, 1 for pchip interpolation.
# scope_module.averager.weight(MIN_NUMBER_OF_RECORDS) # Sets the averaging weight used in exponential moving average.
# scope_module.averager.restart(0) # Set to 1 to restart the averager with the next acquired record. Otherwise 0.

wave_node = uhf.scopes[0].wave # Accessing the wave node of the scope.
scope_module.subscribe(wave_node) # Subscribing to the wave node to receive data.
clockbase = uhf.clockbase() # Get the clock base frequency of the device

#Obtain scope records from the device using an instance of the Scope Module.

def check_scope_record_flags(scope_records, num_records):
    # Loop over all records and print a warning to the console if an error bit in flags has been set.
    num_records = len(scope_records) # Number of records
    for index, record in enumerate(scope_records):
        record_idx = f"{index}/{num_records}" # Record number
        record_flags = record[0]["flags"] # Flag number
        if record_flags & 1:
            print(f"Warning: Scope record {record_idx} flag indicates dataloss.")
        if record_flags & 2:
            print(f"Warning: Scope record {record_idx} indicates missed trigger.")
        if record_flags & 4:
            print(f"Warning: Scope record {record_idx} indicates transfer failure (corrupt data).")
        totalsamples = record[0]["totalsamples"] # Number od samples in the record
        for wave in record[0]["wave"]:
            # Check that the wave in each scope channel contains the expected number of samples.
            assert (len(wave) == totalsamples), f"Scope record {index}/{num_records} size does not match totalsamples."
        
def is_record_clean(record): # Feed in a single record from the scope data
    flags = record[0]["flags"] # Check the flags
    # Check for dataloss (Bit 0), missed trigger (Bit 1), or corrupt data (Bit 2)
    if (flags & 1) or (flags & 2) or (flags & 4):
        return False    
    # Check that the wave in each scope channel contains the expected number of samples
    totalsamples = record[0]["totalsamples"]
    for wave in record[0]["wave"]:
        if len(wave) != totalsamples:
            return False            
    return True

def get_scope_records(scope_module, num_records: int):
    clean_records = []
    # Obtain scope records from the device using an instance of the Scope Module.
    scope_module.raw_module.execute()
    uhf.scopes[0].enable(True)
    session.sync()
    start = time.time()
    timeout = 100000 # [s]
    records = 0
    progress = 0
    # Wait until the Scope Module has received and processed the desired number of records.
    while (records < num_records) or (progress < 1.0):
        # time.sleep(0.1)
        records = scope_module.records()
        progress = scope_module.raw_module.progress()[0]
        # print(f"Scope module has acquired {records} records (requested {num_records}).",end="\r")
        if (time.time() - start) > timeout:
            # Break out of the loop if for some reason we're no longer receiving scope data from the device.
            print(f"\nScope Module did not return {num_records} records after {timeout} s - forcing stop.")
            break
    uhf.scopes[0].enable(True)
    # Read out the scope data from the module.
    data = scope_module.raw_module.read(True)[wave_node.node_info.path]
    # Stop the module; to use it again we need to call execute().
    scope_module.raw_module.finish()
    return data

def get_clean_scope_records(scope_module, num_records: int):
    clean_records = []
    timeout = 100000  # [s]
    start = time.time()
    # Loop until we have accumulated the requested number of clean records
    while len(clean_records) < num_records:
        records_needed = num_records - len(clean_records) # Determine how many records we still need in this cycle
        # Obtain scope records from the device using an instance of the Scope Module.
        scope_module.raw_module.execute()
        uhf.scopes[0].enable(True)
        session.sync()
        records_in_module = 0
        progress = 0
        # Wait until the Scope Module has processed the records for this execution cycle
        while (records_in_module < records_needed) or (progress < 1.0):
            records_in_module = scope_module.records()
            progress = scope_module.raw_module.progress()[0]
            
            if (time.time() - start) > timeout:
                # Break out of the loop if for some reason we're no longer receiving scope data from the device.
                print(f"\nTimeout! Returning {len(clean_records)} clean records after {timeout} s.")
                scope_module.raw_module.finish()
                return clean_records
                
        # Read out the scope data from the module. True flag extracts and clears the buffer.
        raw_data = scope_module.raw_module.read(True)[wave_node.node_info.path]
        # Filter the raw batch and append only clean records
        for record in raw_data:
            if is_record_clean(record):
                clean_records.append(record)
                # Stop appending if we've hit our target midway through a batch
                if len(clean_records) == num_records:
                    break
                        
        # Stop the module; resets it so we can cleanly execute() again if we still need more records.
        scope_module.raw_module.finish()
        
    return clean_records

def get_scope_averager_records(scope_module, AVG_COUNT: int):
    # Obtain scope records from the device using an instance of the Scope Module.
    scope_module.historylength(1)
    scope_module.raw_module.execute()
    uhf.scopes[0].enable(True)
    session.sync()
    start = time.time()
    timeout = 100000 # [s]
    current_count = 0
    # 1. Restart the averager to reset the count to 0 with the next record
    scope_module.averager.restart(1)
    # Wait until the Scope Module has received and processed the desired number of records.
    while current_count < AVG_COUNT:
        time.sleep(0.1)
        current_count = scope_module.averager.count()
        # print(f"Averager count: {current_count} / {AVG_COUNT}", end="\r")
        if (time.time() - start) > timeout:
            # Break out of the loop if for some reason we're no longer receiving scope data from the device.
            print(f"\nAverager did not reach {AVG_COUNT} counts after {timeout} s - forcing stop.")
            break
    uhf.scopes[0].enable(True)
    # Read out the scope data from the module.
    data = scope_module.raw_module.read(True)[wave_node.node_info.path]
    # Stop the module; to use it again we need to call execute().
    scope_module.raw_module.finish()
    check_scope_record_flags(data, 1)
    return data

def to_timestamp(record):
    totalsamples = record[0]["totalsamples"]
    dt = record[0]["dt"]
    timestamp = record[0]["timestamp"]
    triggertimestamp = record[0]["triggertimestamp"]
    t = (np.arange(-totalsamples, 0)*dt) + ((timestamp - triggertimestamp)/float(clockbase))
    return t

def to_frequency(record, scope_time):
    totalsamples = record[0]["totalsamples"]
    scope_rate = clockbase / 2 ** scope_time
    return np.linspace(0, scope_rate / 2, totalsamples)

print("UHF Setup run at :", datetime.datetime.now())

UHF Setup run at : 2026-05-07 11:06:24.569702


# Measurement Program for Sweeping Bilt voltage and recording PSD

In [ ]:
print(f"Measurement started at :", datetime.datetime.now())

# Parameters for voltage sweep
vmin = 0.0 # Minimum voltage in volts
vmax = 0.1 # Maximum voltage in volts
vstep = 0.01 # Voltage step in volts
vmode = 'updown' # Sweep mode: up, down, updown, downup, round-up, round-down

r_series = 1000 # Series resistance in ohms for current to voltage conversion

# Open log file and write header
filename = save_file("260429_v2", "Bilt_i3c4_test_run_v2.dat") # date prefixed filename automatically
if not os.path.exists(filename):
    cols = ["Vset_V", "Vmeas_V", "Imeas_A"]
    pd.DataFrame(columns=cols).to_csv(filename, index=False)

for i in sweep(vmin, vmax, vstep, vmode):
    setvolt_bilt(bilt,position=3,channel=4,voltage=i,step=vstep*0.1,steptime=50)
    time.sleep(5) # Wait for the voltage to stabilize
    voltage, current = measure_bilt(bilt,position=3,channel=4) # Measure voltage and current from Bilt
    # Saving data to file
    bilt_header = {"Vset_V": i, "Vmeas_V": voltage, "Imeas_A": current}
    pd.DataFrame([bilt_header]).to_csv(filename, mode='a', header=False, index=False)

    # # Record PSD at specific applied bias
    data_psd = get_clean_scope_records(scope_module, MIN_NUMBER_OF_RECORDS)
    freq = to_frequency(data_psd[0], SCOPE_TIME)
    # Averaging the PSD data
    psd_sum = np.zeros_like(data_psd[0][0]["wave"][0, :])
    for record in data_psd: 
        psd_sum += record[0]["wave"][0, :]  # Accumulate PSD values
    avgs = len(data_psd)
    average_psd = psd_sum / avgs
    psd_data = pd.DataFrame({'Frequency_Hz': freq, 'PSD_V2perHz': average_psd})
    psd_filename= f"260429_Noise_spectrum_at_{i:.3f}V_27.5kHz_{int(SCOPE_LENGTH)}_{int(MIN_NUMBER_OF_RECORDS)}_records_testing_v2.dat"
    create_file(psd_data,"260429_v2\\PSD",psd_filename)

print(f"Measurement ended at :", datetime.datetime.now())    

# Measurement Program for Sweeping Bilt, measureing PSD and TT

In [8]:
# --- Helper Function for the Background Thread ---
def record_dmm_trace(dmm, filename, stop_event):
    dmm_start_time = time.time()
    # Loop runs as long as the main thread hasn't triggered the stop_event
    while not stop_event.is_set():
        try:
            dmm_time = time.time() - dmm_start_time
            voltage = dmm.query_ascii_values("MEAS:VOLT:DC?")[0]
            # Saving data to file
            dmm_header = {"Time_s": dmm_time, "Vmeas_V": voltage}
            pd.DataFrame([dmm_header]).to_csv(filename, mode='a', header=False, index=False)
        except Exception as e:
            break

# --- Main Script ---
print(f"Measurement started at :", datetime.datetime.now())

# Parameters for voltage sweep
vmin = 0.05
vmax = 0.05
vstep = 0.05
vmode = 'up' # Sweep mode: up, down, updown, downup, round-up, round-down

r_series = 1000 

# Open log file and write header
main_filename = save_file("260506_test\\BILT", "Bilt_i2c3_13.30kOhm_200_gain_1MOhm_test.dat") 
if not os.path.exists(main_filename):
    cols = ["Vset_V", "Vmeas_V", "Imeas_A"]
    pd.DataFrame(columns=cols).to_csv(main_filename, index=False)

# Voltage sweep loop
for i in sweep(vmin, vmax, vstep, vmode):
    setvolt_bilt(bilt, position=2, channel=3, voltage=i, step=vstep*0.1, steptime=50)
    print(f"Bilt set to {i:.3f} V at {datetime.datetime.now()}", end="\r")
    time.sleep(5) # Wait for the voltage to stabilize and scope to be ready
    voltage, current = measure_bilt(bilt, position=2, channel=3)
    # Saving data to file
    bilt_header = {"Vset_V": i, "Vmeas_V": voltage, "Imeas_A": current}
    pd.DataFrame([bilt_header]).to_csv(main_filename, mode='a', header=False, index=False)
    print(f"Bilt set to {i:.3f} V at {datetime.datetime.now()}", end="\r")
    # --- 1. PREPARE DMM FILE ---
    dmm_filename = save_file("260506_test\\TT", f"DMM_time_trace_at_{i:.3f}V_13.30kOhm_200_gain_1MOhm_test.dat") 
    if not os.path.exists(dmm_filename):
        cols = ["Time_s", "Vmeas_V"]
        pd.DataFrame(columns=cols).to_csv(dmm_filename, index=False)

    # --- 2. START BACKGROUND THREAD FOR DMM ---
    stop_dmm_event = threading.Event()
    dmm_thread = threading.Thread(
        target=record_dmm_trace, 
        args=(dmm25, dmm_filename, stop_dmm_event)
    )
    dmm_thread.start() # DMM starts recording simultaneously now

    # --- 3. RECORD PSD (Main Thread) ---
    data_psd = get_clean_scope_records(scope_module, MIN_NUMBER_OF_RECORDS)
    
    # --- 4. STOP DMM THREAD ---
    # Once PSD records are acquired, signal the DMM loop to break
    stop_dmm_event.set()
    dmm_thread.join() # Wait a fraction of a second for the DMM loop to safely finish its last write

    # --- 5. PROCESS AND SAVE PSD DATA ---
    freq = to_frequency(data_psd[0], SCOPE_TIME)
    
    # Averaging the PSD data
    psd_sum = np.zeros_like(data_psd[0][0]["wave"][0, :])
    for record in data_psd: 
        psd_sum += record[0]["wave"][0, :]  
        
    avgs = len(data_psd)
    average_psd = psd_sum / avgs
    
    psd_data = pd.DataFrame({'Frequency_Hz': freq, 'PSD_V2perHz': average_psd})
    psd_filename = f"260506_Noise_spectrum_at_{i:.3f}V_13.30kOhm_200_gain_1MOhm_27.7kHz_524288samples_20averaged_test.dat"
    create_file(psd_data, "260506_test\\PSD", psd_filename)

print(f"Measurement ended at :", datetime.datetime.now())

Measurement started at : 2026-05-06 20:42:56.758990
Measurement ended at : 2026-05-06 20:49:34.468065


# Measurement program for sweeping bilt Voltage and measuring Vdiff

In [53]:
# Open Pyvisa Resource Manager and connect to insturements at 14T cryostat
rm = pyvisa.ResourceManager()
bilt = rm.open_resource("TCPIP0::192.168.150.123::5025::SOCKET", timeout=5000, write_termination='\n', read_termination='\n')
print(f"bilt            : {bilt.query('*IDN?')}")
dmm25 = rm.open_resource("TCPIP0::192.168.150.124::gpib0,25::INSTR", timeout=5000, write_termination='\n', read_termination='\n')
print(f"dmm25           : {dmm25.query('*IDN?')}")

print(f"Measurement started at :", datetime.datetime.now())

setvolt_bilt(bilt,position=2,channel=3,voltage=0,step=vstep*0.1,steptime=50)
time.sleep(5) # Wait for the voltage to stabilize before starting the sweep
dmm25.write("CALCulate:STATe ON")

# Parameters for voltage sweep
vmin = -0.5 # Minimum voltage in volts
vmax = 0.5 # Maximum voltage in volts
vstep = 0.001 # Voltage step in volts
vmode = 'updown' # Sweep mode: up, down, updown, downup, round-up, round-down

r_series = 1e6 # Series resistance in ohms for current to voltage conversion
gain = 200 # Gain of the amplifier used for current to voltage conversion

setvolt_bilt(bilt,position=2,channel=3,voltage=vmin,step=vstep*0.1,steptime=50)
time.sleep(5) # Wait for the voltage to stabilize before starting the sweep

# Open log file and write header
filename = save_file("260508", "IV_200gain_1MOhm_4K_0T_v3.dat") # date prefixed filename automatically
if not os.path.exists(filename):
    cols = ["Vset_V", "Vin_V", "Rs_Ohm", "Iin_A", "Vmeas_V", "Gain", "Vdiff_V"]
    pd.DataFrame(columns=cols).to_csv(filename, index=False)

for i in sweep(vmin, vmax, vstep, vmode):
    setvolt_bilt(bilt,position=2,channel=3,voltage=i,step=vstep*0.1,steptime=50)
    time.sleep(0.3) # Wait for the voltage to stabilize
    Vin_V, dummy = measure_bilt(bilt,position=2,channel=3) # Measure voltage and current from Bilt
    current = Vin_V / r_series
    Vmeas_V = dmm25.query_ascii_values("MEAS:VOLT:DC?")[0]
    Vdiff_V = Vmeas_V / gain
    # Saving data to file
    bilt_header = {"Vset_V": i, "Vin_V": Vin_V, "Rs_Ohm": r_series, "Iin_A": current, "Vmeas_V": Vmeas_V, "Gain": gain, "Vdiff_V": Vdiff_V}
    pd.DataFrame([bilt_header]).to_csv(filename, mode='a', header=False, index=False)

setvolt_bilt(bilt,position=2,channel=3,voltage=0,step=vstep*0.1,steptime=50)
time.sleep(5)

print(f"Measurement ended at :", datetime.datetime.now())    

bilt            : 2142,"ITEST BE2142C/12V 15mA DC-SOURCE/SN06-013 LC2002 VL446\240"
dmm25           : HEWLETT-PACKARD,34401A,0,11-5-2
Measurement started at : 2026-05-08 17:28:16.182518
Measurement ended at : 2026-05-08 17:56:33.177777


# Just for recording PSD manually after averaging

In [32]:
# # Record PSD at specific applied bias
data_psd = get_clean_scope_records(scope_module, 30)
freq = to_frequency(data_psd[0], SCOPE_TIME)
# Averaging the PSD data
psd_sum = np.zeros_like(data_psd[0][0]["wave"][0, :])
for record in data_psd: 
    psd_sum += record[0]["wave"][0, :]  # Accumulate PSD values
avgs = len(data_psd)
average_psd = psd_sum / avgs

psd_data = pd.DataFrame({'Frequency_Hz': freq, 'PSD_V2perHz': average_psd})
psd_filename= f"260507_Noise_spectrum_at_0.000V_98.00kOhm_880kHz_65536_30averaged_test_v1.dat"
create_file(psd_data,"260507_test",psd_filename)

In [51]:
dmm25.write("CALCulate:STATe ON")
# dmm25.query_ascii_values("MEAS:VOLT:DC?")[0]

19

In [52]:
dmm25.query("CALCulate:STATe?")

'1'